In [ ]:
import csv
import json
import urllib.request

# Download the raw CSV from the root of your GitHub repository.
url = "https://raw.githubusercontent.com/jyotikeshri99999/Capstone-Project/main/appointments.csv"
csv_path = "appointments.csv"

with urllib.request.urlopen(url) as response:
    csv_text = response.read().decode("utf-8-sig")

def parse_row(row):
    """Return a copy of the row with numeric fields converted."""
    converted = row.copy()
    converted["age"] = int(converted["age"])
    converted["fee_inr"] = int(converted["fee_inr"])
    converted["prior_no_shows"] = int(converted["prior_no_shows"])
    converted["distance_km"] = float(converted["distance_km"])
    return converted

accepted = []
rejected = []

# DictReader reads the CSV text as rows of dictionaries.
reader = csv.DictReader(csv_text.splitlines())

print("Header:", reader.fieldnames)

for row in reader:
    try:
        accepted.append(parse_row(row))
    except ValueError:
        rejected.append(row)

rows_read = len(accepted) + len(rejected)

print("Rows read:", rows_read)
print("Accepted:", len(accepted))
print("Rejected:", len(rejected))

# Save rows that could not be converted.
with open("rejected_rows.json", "w", encoding="utf-8") as file:
    json.dump(rejected, file, indent=2)

print("Saved rejected_rows.json")

Header: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Rows read: 1020
Accepted: 988
Rejected: 32
Saved rejected_rows.json


In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/jyotikeshri99999/Capstone-Project/main/appointments.csv"

# Read the original file again; don't use Part 1.1's accepted/rejected lists.
df = pd.read_csv(url)

print("Raw shape:", df.shape)
print("\nColumn information:")
df.info()

print("\nMissing values by column:")
print(df.isnull().sum())

print("\nExact duplicate rows:", df.duplicated().sum())
print(
    "Distinct raw booking_channel values:",
    df["booking_channel"].nunique(dropna=True)
)

# (a) Drop exact duplicate rows.
df = df.drop_duplicates().copy()
print("\nRows after dropping exact duplicates:", len(df))

# (b) Clean booking_channel labels.
df["booking_channel"] = df["booking_channel"].str.strip().str.title()
print(
    "Distinct booking_channel values after cleaning:",
    df["booking_channel"].nunique(dropna=True)
)
print("Rows after cleaning booking_channel:", len(df))

# (c) Convert dates.
df["appt_date"] = pd.to_datetime(df["appt_date"], errors="coerce")
df["booked_on"] = pd.to_datetime(df["booked_on"])
print("Rows after converting dates:", len(df))

# (d) Drop rows with a missing appointment date or status.
df = df.dropna(subset=["appt_date", "status"]).copy()
print("Rows after dropping missing appt_date or status:", len(df))

Raw shape: (1020, 14)

Column information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   object 
 1   patient_id       1020 non-null   object 
 2   clinic_id        1020 non-null   object 
 3   booked_on        1020 non-null   object 
 4   appt_date        1019 non-null   object 
 5   age              1008 non-null   float64
 6   gender           1020 non-null   object 
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   object 
 10  visit_type       1020 non-null   object 
 11  reminder_sent    1020 non-null   object 
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   object 
dtypes: float64(2), int64(2), object(10)
memory usage: 111.7+ KB

Missing values by column:
appointment_id      0


In [ ]:
import numpy as np

# (a) Mark impossible ages as missing.
invalid_age_mask = (df["age"] < 1) | (df["age"] > 100)
print("Invalid ages:", int(invalid_age_mask.sum()))
df.loc[invalid_age_mask, "age"] = np.nan

# (b) Fill missing ages and distances with their medians.
print("Missing ages before filling:", int(df["age"].isna().sum()))
print("Median age:", df["age"].median())
df["age"] = df["age"].fillna(df["age"].median()).round().astype(int)

print("Missing distances before filling:", int(df["distance_km"].isna().sum()))
print("Median distance_km:", df["distance_km"].median())
df["distance_km"] = df["distance_km"].fillna(df["distance_km"].median())

# (c) Calculate IQR fences, then cap values outside the fences.
def cap_outliers(column_name):
    q1 = df[column_name].quantile(0.25)
    q3 = df[column_name].quantile(0.75)
    iqr = q3 - q1
    lower_fence = q1 - 1.5 * iqr
    upper_fence = q3 + 1.5 * iqr

    capped_count = int(
        ((df[column_name] < lower_fence) |
         (df[column_name] > upper_fence)).sum()
    )

    df[column_name] = np.clip(
        df[column_name],
        lower_fence,
        upper_fence
    )

    print(
        f"{column_name}: Q1={q1}, Q3={q3}, "
        f"IQR={iqr}, lower fence={lower_fence}, "
        f"upper fence={upper_fence}, values capped={capped_count}"
    )

cap_outliers("distance_km")
cap_outliers("fee_inr")

# (d) Round the capped columns as specified.
df["distance_km"] = df["distance_km"].round(1)
df["fee_inr"] = df["fee_inr"].round().astype(int)

print("\nMedian-filled age:", df["age"].median())
print("Largest fee after capping:", df["fee_inr"].max())
print("Missing ages remaining:", int(df["age"].isna().sum()))
print("Missing distances remaining:", int(df["distance_km"].isna().sum()))

Invalid ages: 5
Missing ages before filling: 17
Median age: 48.0
Missing distances before filling: 20
Median distance_km: 8.25
distance_km: Q1=4.625, Q3=13.0, IQR=8.375, lower fence=-7.9375, upper fence=25.5625, values capped=26
fee_inr: Q1=500.0, Q3=787.5, IQR=287.5, lower fence=68.75, upper fence=1218.75, values capped=3

Median-filled age: 48.0
Largest fee after capping: 1219
Missing ages remaining: 0
Missing distances remaining: 0


In [ ]:
# Create the two derived columns.
df["lead_days"] = (df["appt_date"] - df["booked_on"]).dt.days
df["no_show"] = (df["status"] == "No-Show").astype(int)

# Convert the four requested columns into a NumPy array.
X = df[["age", "distance_km", "fee_inr", "lead_days"]].to_numpy()

# (a) Mean of each column.
column_means = np.mean(X, axis=0)
print("Means [age, distance_km, fee_inr, lead_days]:", column_means)

# (b) Count and share of appointments with lead_days > 14.
long_lead_mask = df["lead_days"].to_numpy() > 14
print("Appointments with lead_days > 14:", int(long_lead_mask.sum()))
print("Share with lead_days > 14:", round(long_lead_mask.mean() * 100, 1), "%")

# (c) Distance z-scores and count where absolute z-score > 2.
distance = X[:, 1]
distance_z = (distance - np.mean(distance)) / np.std(distance)
print("Rows with |distance z-score| > 2:", int((np.abs(distance_z) > 2).sum()))

# Show the overall no-show count and rate for comparison.
print("No-shows:", int(df["no_show"].sum()), "out of", len(df))
print("No-show rate:", round(df["no_show"].mean() * 100, 1), "%")

Means [age, distance_km, fee_inr, lead_days]: [ 48.17373737   9.50090909 635.71414141  11.81919192]
Appointments with lead_days > 14: 296
Share with lead_days > 14: 29.9 %
Rows with |distance z-score| > 2: 53
No-shows: 218 out of 990
No-show rate: 22.0 %


In [ ]:
import requests
import pandas as pd

clinics_url = (
    "https://raw.githubusercontent.com/"
    "jyotikeshri99999/Capstone-Project/main/clinics.csv"
)

clinics = pd.read_csv(clinics_url)
temperatures = []

for _, clinic in clinics.iterrows():
    params = {
        "latitude": clinic["latitude"],
        "longitude": clinic["longitude"],
        "hourly": "temperature_2m",
        "forecast_days": 1
    }

    temp_c = None
    last_reason = "Unknown error"

    # Try once, then retry once if the request or response fails.
    for attempt in range(2):
        try:
            response = requests.get(
                "https://api.open-meteo.com/v1/forecast",
                params=params,
                timeout=10
            )

            if response.status_code != 200:
                last_reason = f"HTTP status {response.status_code}"
            else:
                data = response.json()
                hourly = data.get("hourly", {})
                readings = hourly.get("temperature_2m")

                if readings:
                    temp_c = readings[0]
                    break
                else:
                    last_reason = "Hourly temperature data is missing or empty"

        except requests.exceptions.RequestException as error:
            last_reason = str(error)
        except (ValueError, TypeError) as error:
            last_reason = f"Invalid response: {error}"

    if temp_c is None:
        print(f"{clinic['clinic_name']}: weather lookup failed — {last_reason}")

    temperatures.append(temp_c)

clinics["temp_c"] = temperatures
clinics.to_csv("clinics_enriched.csv", index=False)

print("\nSaved clinics_enriched.csv")
print("Rows:", len(clinics))
print(clinics[["clinic_name", "temp_c"]])


Saved clinics_enriched.csv
Rows: 8
                clinic_name  temp_c
0     Kothrud Family Clinic    19.2
1      Hadapsar Care Centre    19.5
2    Sitabuldi Health Point    18.6
3      Malviya Nagar Clinic    14.4
4     Vaishali Nagar Clinic    14.4
5  Edappally Medical Centre    27.9
6        Vijay Nagar Clinic    16.6
7    Gomti Nagar Care Point    14.0


In [ ]:
import pandas as pd

columns = [
    "appointment_id", "patient_id", "clinic_id", "booked_on", "appt_date",
    "age", "gender", "distance_km", "fee_inr", "booking_channel",
    "visit_type", "reminder_sent", "prior_no_shows", "status",
    "lead_days", "no_show"
]

# Sort and keep the required column order.
clean = df.sort_values("appointment_id").copy()

# Format dates as YYYY-MM-DD text.
clean["booked_on"] = clean["booked_on"].dt.strftime("%Y-%m-%d")
clean["appt_date"] = clean["appt_date"].dt.strftime("%Y-%m-%d")
clean = clean[columns]

# Check the required export conditions.
print("Unique appointment IDs:", clean["appointment_id"].nunique())
print("Appointment rows:", len(clean))
print("Missing values by column:")
print(clean.isna().sum())
print("Total missing values:", int(clean.isna().sum().sum()))

# Save the clean CSV.
clean.to_csv("appointments_clean.csv", index=False)
print("\nSaved appointments_clean.csv")

def quote_sql(value):
    """Format a text value for this fixed dataset's SQL inserts."""
    return "'" + str(value).replace("'", "''") + "'"

def sql_number(value):
    """Format a numeric value, or use SQL NULL if it is missing."""
    if pd.isna(value):
        return "NULL"
    return str(value)

def sql_decimal(value, places):
    """Format a decimal value, or use SQL NULL if it is missing."""
    if pd.isna(value):
        return "NULL"
    return f"{float(value):.{places}f}"

with open("load_data.sql", "w", encoding="utf-8") as sql_file:

    # Clinic inserts must come first because appointments reference clinics.
    for _, row in clinics.iterrows():
        values = [
            quote_sql(row["clinic_id"]),
            quote_sql(row["clinic_name"]),
            quote_sql(row["city"]),
            quote_sql(row["tier"]),
            sql_decimal(row["latitude"], 4),
            sql_decimal(row["longitude"], 4),
            sql_decimal(row["temp_c"], 1)
        ]
        sql_file.write(
            "INSERT INTO clinics VALUES (" + ",".join(values) + ");\n"
        )

    # Appointment inserts.
    for _, row in clean.iterrows():
        values = [
            quote_sql(row["appointment_id"]),
            quote_sql(row["patient_id"]),
            quote_sql(row["clinic_id"]),
            quote_sql(row["booked_on"]),
            quote_sql(row["appt_date"]),
            sql_number(row["age"]),
            quote_sql(row["gender"]),
            sql_decimal(row["distance_km"], 1),
            sql_number(row["fee_inr"]),
            quote_sql(row["booking_channel"]),
            quote_sql(row["visit_type"]),
            quote_sql(row["reminder_sent"]),
            sql_number(row["prior_no_shows"]),
            quote_sql(row["status"]),
            sql_number(row["lead_days"]),
            sql_number(row["no_show"])
        ]
        sql_file.write(
            "INSERT INTO appointments VALUES (" + ",".join(values) + ");\n"
        )

print("Saved load_data.sql")
print("Expected INSERT statements:", len(clinics) + len(clean))

Unique appointment IDs: 990
Appointment rows: 990
Missing values by column:
appointment_id     0
patient_id         0
clinic_id          0
booked_on          0
appt_date          0
age                0
gender             0
distance_km        0
fee_inr            0
booking_channel    0
visit_type         0
reminder_sent      0
prior_no_shows     0
status             0
lead_days          0
no_show            0
dtype: int64
Total missing values: 0

Saved appointments_clean.csv
Saved load_data.sql
Expected INSERT statements: 998
